# NoMC Ablation

## Imports and Setup

In [ ]:
# Mount Drive and set up directories
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/GNN_MEA/'
VICTIM_DIR = os.path.join(BASE_DIR, 'victim_models')
EXPLAINER_DIR = os.path.join(BASE_DIR, 'explainers')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
ARCHITECTURES = ['GCN']

In [ ]:
!pip install torch_geometric

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, global_mean_pool
from torch_geometric.transforms import BaseTransform, OneHotDegree
from torch_geometric.explain import Explainer, PGExplainer, GNNExplainer
from sklearn.model_selection import train_test_split
import numpy as np
import copy
import json
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Dataset Loading

In [ ]:
# Load only ablation datasets
DATASET_NAMES = ['AIDS', 'NCI1', 'Tox21_AhR_training']

datasets = {}
for name in DATASET_NAMES:
    ds = TUDataset(root=f'data/{name}', name=name)
    datasets[name] = ds
    assert ds[0].x is not None, f"{name}: features still None!"
    print(f"{name}: {len(ds)} graphs, {ds.num_classes} classes, "
          f"feature dim = {ds[0].x.shape[1]}")

In [ ]:
# Stratified 60% train / 20% shadow / 20% test split
def split_dataset(dataset, seed=42):
    labels = [data.y.item() for data in dataset]
    train_idx, remaining_idx = train_test_split(
        range(len(dataset)), test_size=0.4,
        stratify=labels, random_state=seed)
    remaining_labels = [labels[i] for i in remaining_idx]
    shadow_idx, test_idx = train_test_split(
        remaining_idx, test_size=0.5,
        stratify=remaining_labels, random_state=seed)
    return train_idx, shadow_idx, test_idx

def get_feature_dim(dataset):
    return dataset[0].x.shape[1]

# Compute splits for all datasets
dataset_splits = {}
for name in DATASET_NAMES:
    dataset_splits[name] = dict(zip(
        ['train', 'shadow', 'test'],
        split_dataset(datasets[name])))

## Model Architectures

In [ ]:
# 3-layer GNN architectures
class GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = self.classifier(x)
        return x

class GAT(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes, dropout=0.5):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden_dim // 8, heads=8)
        self.conv2 = GATConv(hidden_dim, hidden_dim // 8, heads=8)
        self.conv3 = GATConv(hidden_dim, hidden_dim // 8, heads=8)
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = self.classifier(x)
        return x

class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.conv3 = SAGEConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = self.classifier(x)
        return x

MODEL_CLASSES = {'GCN': GCN, 'GAT': GAT, 'GraphSAGE': GraphSAGE}

## Victim Model Loading

In [ ]:
# Load saved victim model from Drive
def load_victim(name, arch, dataset, device='cuda'):
    save_path = os.path.join(VICTIM_DIR, arch, f'{name}_victim.pt')
    checkpoint = torch.load(save_path, map_location=device, weights_only=False)
    ModelClass = MODEL_CLASSES[arch]
    model = ModelClass(
        checkpoint['in_dim'],
        checkpoint['config']['hidden_dim'],
        checkpoint['num_classes']
    ).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model, checkpoint

# Black-box query: get victim's prediction for a single graph
def get_prediction(model, data, device='cuda'):
    model.eval()
    data = data.to(device)
    batch = torch.zeros(data.num_nodes, dtype=torch.long, device=device)
    with torch.no_grad():
        pred = model(data.x, data.edge_index, batch).argmax(dim=1).item()
    return pred

# Load GCN victims for ablation datasets only
victim_models = {}
for name in DATASET_NAMES:
    ds = datasets[name]
    for arch in ARCHITECTURES:
        key = f"{name}_{arch}"
        path = os.path.join(VICTIM_DIR, arch, f'{name}_victim.pt')
        if os.path.exists(path):
            model, ckpt = load_victim(name, arch, ds, device)
            victim_models[key] = model
            print(f"  Loaded {key}: acc={ckpt['accuracy']:.4f}")
        else:
            print(f"  MISSING: {key}")

## Explainer Setup

In [ ]:
# Get explanation: handles edge masks (PGExp) and node masks (GNNExp)
def get_explanation(explainer, model, data, device='cuda'):
    data = data.to(device)
    batch = torch.zeros(data.num_nodes, dtype=torch.long, device=device)
    target = model(data.x, data.edge_index, batch).argmax(dim=1)

    explanation = explainer(data.x, data.edge_index,
                            target=target, batch=batch)

    if hasattr(explanation, 'node_mask') and explanation.node_mask is not None:
        node_mask = explanation.node_mask
        node_importance = node_mask.mean(dim=1) if node_mask.dim() > 1 else node_mask
        threshold = node_importance.median()
        important_nodes = (node_importance > threshold).nonzero(as_tuple=True)[0]
        src, dst = data.edge_index
        edge_mask_bool = (torch.isin(src, important_nodes) &
                          torch.isin(dst, important_nodes))
        important_edges = edge_mask_bool.nonzero(as_tuple=True)[0]
        return important_nodes.cpu(), important_edges.cpu(), node_importance.cpu()

    elif hasattr(explanation, 'edge_mask') and explanation.edge_mask is not None:
        edge_mask = explanation.edge_mask
        threshold = edge_mask.median()
        important_edges = (edge_mask > threshold).nonzero(as_tuple=True)[0]
        important_nodes = torch.unique(
            data.edge_index[:, important_edges].flatten())
        return important_nodes.cpu(), important_edges.cpu(), edge_mask.cpu()

    else:
        raise ValueError("No mask found in explanation")

# Instance-level explainers use a COPY of the model (don't corrupt original)
def create_gnn_explainer(model):
    model_copy = copy.deepcopy(model)
    for param in model_copy.parameters():
        param.requires_grad_(True)
    return Explainer(
        model=model_copy,
        algorithm=GNNExplainer(epochs=100, lr=0.01),
        explanation_type='phenomenon',
        node_mask_type='object',
        edge_mask_type=None,
        model_config=dict(
            mode='multiclass_classification',
            task_level='graph',
            return_type='raw',
        ),
    )

## Helper Functions

In [ ]:
# Get undirected edge set from edge_index
def get_edge_set(edge_index):
    edges = set()
    for k in range(edge_index.shape[1]):
        u, v = edge_index[0, k].item(), edge_index[1, k].item()
        edges.add((min(u, v), max(u, v)))
    return edges

# Flip an edge in list (remove if exists, add if absent)
def flip_edge_in_list(edge_list, existing, u, v):
    if (u, v) in existing:
        edge_list = [e for e in edge_list
                     if not ((e[0] == u and e[1] == v) or
                             (e[0] == v and e[1] == u))]
        existing.discard((u, v))
    else:
        edge_list.append([u, v])
        edge_list.append([v, u])
        existing.add((u, v))
    return edge_list, existing

## Phase 1: Exhaustive δ=1 Boundary Search (Shared)

In [ ]:
# Phase 1: Cheap exhaustive delta=1 search
# (shared computation for both ablations)

def find_boundary_pairs_exhaustive(dataset, shadow_idx, model, device='cuda'):
    boundary_pairs = []
    non_boundary_idx = []

    for i in shadow_idx:
        data = dataset[i]
        orig_pred = get_prediction(model, data, device)

        existing = get_edge_set(data.edge_index)

        found = False
        for (u, v) in existing:
            flipped = data.clone()
            edge_list = flipped.edge_index.t().tolist()
            edge_list = [e for e in edge_list
                         if not ((e[0] == u and e[1] == v) or
                                 (e[0] == v and e[1] == u))]

            if len(edge_list) == 0:
                continue

            flipped.edge_index = torch.tensor(edge_list, dtype=torch.long).t()
            new_pred = get_prediction(model, flipped, device)

            if new_pred != orig_pred:
                orig = data.clone()
                orig.y = torch.tensor([orig_pred])
                flipped.y = torch.tensor([new_pred])
                boundary_pairs.append((orig, flipped))
                found = True
                break

        if not found:
            non_boundary_idx.append(i)

    print(f"  Phase 1: {len(boundary_pairs)} boundary pairs, "
          f"{len(non_boundary_idx)} non-boundary graphs")
    return boundary_pairs, non_boundary_idx

## Phase 2: MC Sensitivity Estimation and Boundary Search

Needed by NoMC (calls `boundary_search` with `n_mc=1`).

In [ ]:
# Build candidate edges around explanation subgraph (Eq. 15)
def build_candidates(data, important_nodes, carry_over=None, n_random=10):
    n = data.num_nodes
    imp_set = set(important_nodes.tolist())

    neighbors = set()
    for k in range(data.edge_index.shape[1]):
        u, v = data.edge_index[0, k].item(), data.edge_index[1, k].item()
        if u in imp_set and v not in imp_set:
            neighbors.add(v)
        if v in imp_set and u not in imp_set:
            neighbors.add(u)

    candidates = set()

    for u in imp_set:
        for v in imp_set:
            if u < v:
                candidates.add((u, v))

    for u in imp_set:
        for v in neighbors:
            candidates.add((min(u, v), max(u, v)))

    if carry_over is not None:
        candidates.update(carry_over)

    for _ in range(n_random):
        u, v = np.random.randint(0, n, size=2)
        if u != v:
            candidates.add((min(u, v), max(u, v)))

    return candidates

# MC edge sensitivity estimation (Eq. 8-9)
# Measures whether flipping edge e pushes prediction away from orig_pred
def estimate_edge_sensitivity(data, edge, model, orig_pred,
                              n_samples=5, p_flip=0.05, device='cuda'):
    u, v = edge
    scores = []

    edge_list_orig = data.edge_index.t().tolist()
    existing_orig = get_edge_set(data.edge_index)

    for _ in range(n_samples):
        perturbed_edges = list(edge_list_orig)
        perturbed_existing = set(existing_orig)

        for k in range(data.num_nodes):
            for l in range(k + 1, data.num_nodes):
                if (k, l) == (u, v):
                    continue
                if np.random.random() < p_flip:
                    perturbed_edges, perturbed_existing = flip_edge_in_list(
                        perturbed_edges, perturbed_existing, k, l)

        if len(perturbed_edges) == 0:
            continue

        base = data.clone()
        base.edge_index = torch.tensor(perturbed_edges, dtype=torch.long).t()
        pred_base = get_prediction(model, base, device)

        flipped_edges = list(perturbed_edges)
        flipped_existing = set(perturbed_existing)
        flipped_edges, flipped_existing = flip_edge_in_list(
            flipped_edges, flipped_existing, u, v)

        if len(flipped_edges) == 0:
            continue

        flipped = data.clone()
        flipped.edge_index = torch.tensor(flipped_edges, dtype=torch.long).t()
        pred_flipped = get_prediction(model, flipped, device)

        # Multi-class compatible: rewards any change away from orig_pred
        z = int(pred_flipped != orig_pred) - int(pred_base != orig_pred)
        scores.append(z)

    return np.mean(scores) if scores else 0.0

# Iterative boundary search for one graph (Algorithm 2, edge-only)
def boundary_search(data, model, explainer, max_iters=10, top_k=3,
                    n_mc=5, delta_max=20, p_flip=0.05, n_random=10,
                    device='cuda'):
    orig_pred = get_prediction(model, data, device)
    current = data.clone()
    orig_edges = get_edge_set(data.edge_index)
    carry_over = None

    for t in range(max_iters):
        cur_pred = get_prediction(model, current, device)
        cur_edges = get_edge_set(current.edge_index)
        delta_e = len(orig_edges.symmetric_difference(cur_edges))

        # Success: prediction changed within distance budget
        if cur_pred != orig_pred and delta_e <= delta_max:
            current.y = torch.tensor([cur_pred])
            return current, delta_e

        if delta_e >= delta_max:
            break

        # Get explanation for current graph state
        imp_nodes, imp_edges, mask = get_explanation(
            explainer, model, current, device)

        candidates = build_candidates(current, imp_nodes, carry_over, n_random)

        # Estimate sensitivity: pass orig_pred (works for binary and multi-class)
        sensitivities = {}
        for edge in candidates:
            g_hat = estimate_edge_sensitivity(
                current, edge, model, orig_pred, n_mc, p_flip, device)
            sensitivities[edge] = g_hat

        sorted_edges = sorted(sensitivities.items(), key=lambda x: -x[1])
        top_edges = [e for e, s in sorted_edges[:top_k] if s > 0]

        if len(top_edges) == 0:
            break

        edge_list = current.edge_index.t().tolist()
        existing = get_edge_set(current.edge_index)
        for (u, v) in top_edges:
            edge_list, existing = flip_edge_in_list(edge_list, existing, u, v)

        if len(edge_list) > 0:
            current.edge_index = torch.tensor(edge_list, dtype=torch.long).t()

        carry_over = set(e for e, s in sorted_edges[:5] if s > 0)

    return None, 0

## NoMC Ablation Methods

In [ ]:
# Build RANDOM candidate edges (no explanation, for NoExp ablation)
def build_candidates_random(data, n_candidates=35, carry_over=None, n_random=10):
    n = data.num_nodes
    candidates = set()

    # Random edges to match typical explanation-guided candidate set size
    while len(candidates) < n_candidates:
        u, v = np.random.randint(0, n, size=2)
        if u != v:
            candidates.add((min(u, v), max(u, v)))

    if carry_over is not None:
        candidates.update(carry_over)

    return candidates

# Boundary search WITHOUT explanation guidance (NoExp ablation)
def boundary_search_no_exp(data, model, max_iters=10, top_k=3,
                           n_mc=5, delta_max=10, p_flip=0.05,
                           n_candidates=35, device='cuda'):
    orig_pred = get_prediction(model, data, device)
    current = data.clone()
    orig_edges = get_edge_set(data.edge_index)
    carry_over = None

    for t in range(max_iters):
        cur_pred = get_prediction(model, current, device)
        cur_edges = get_edge_set(current.edge_index)
        delta_e = len(orig_edges.symmetric_difference(cur_edges))

        if cur_pred != orig_pred and delta_e <= delta_max:
            current.y = torch.tensor([cur_pred])
            return current, delta_e

        if delta_e >= delta_max:
            break

        num_classes = data.y.max().item() + 1 if data.y.max() > 1 else 2
        target_label = 1 - cur_pred if num_classes == 2 else (cur_pred + 1) % num_classes

        # Random candidates instead of explanation-guided
        candidates = build_candidates_random(current, n_candidates, carry_over)

        sensitivities = {}
        for edge in candidates:
            g_hat = estimate_edge_sensitivity(
                current, edge, model, target_label, n_mc, p_flip, device)
            sensitivities[edge] = g_hat

        sorted_edges = sorted(sensitivities.items(), key=lambda x: -x[1])
        top_edges = [e for e, s in sorted_edges[:top_k] if s > 0]

        if len(top_edges) == 0:
            break

        edge_list = current.edge_index.t().tolist()
        existing = get_edge_set(current.edge_index)
        for (u, v) in top_edges:
            edge_list, existing = flip_edge_in_list(edge_list, existing, u, v)

        if len(edge_list) > 0:
            current.edge_index = torch.tensor(edge_list, dtype=torch.long).t()

        carry_over = set(e for e, s in sorted_edges[:5] if s > 0)

    return None, 0

In [ ]:
# NoMC method: Phase 1 (same) + Phase 2 with n_mc=1
def method_boundary_no_mc(dataset, shadow_idx, model, explainer, n_samples,
                          boundary_pairs_p1=None, non_boundary_idx=None,
                          device='cuda'):
    if boundary_pairs_p1 is None:
        boundary_pairs_p1, non_boundary_idx = find_boundary_pairs_exhaustive(
            dataset, shadow_idx, model, device)

    # Phase 1 results (identical to full method)
    train_data = []
    for orig, flipped in boundary_pairs_p1:
        train_data.append(orig)
        train_data.append(flipped)

    # Phase 2 with n_mc=1 (single sample, no averaging)
    if len(train_data) < n_samples and non_boundary_idx and len(non_boundary_idx) > 0:
        n_needed = (n_samples - len(train_data)) // 2
        pairs_p2 = []
        n_searched = 0

        for i in non_boundary_idx:
            if len(pairs_p2) >= n_needed:
                break
            data = dataset[i]
            orig_pred = get_prediction(model, data, device)
            n_searched += 1

            # KEY DIFFERENCE: n_mc=1 instead of n_mc=5
            result, d_e = boundary_search(
                data, model, explainer, n_mc=1, device=device)

            if result is not None:
                orig = data.clone()
                orig.y = torch.tensor([orig_pred])
                pairs_p2.append((orig, result))

        print(f"  NoMC Phase 2: {len(pairs_p2)} pairs "
              f"from {n_searched} graphs")
        for orig, flipped in pairs_p2:
            train_data.append(orig)
            train_data.append(flipped)

    print(f"  NoMC total: {len(train_data[:n_samples])} samples")
    return train_data[:n_samples]

## Training and Evaluation

In [ ]:
# Surrogate training (fixed HPs, same arch as victim)
def train_surrogate(train_data, in_dim, num_classes, arch='GCN', device='cuda'):
    ModelClass = MODEL_CLASSES[arch]
    model = ModelClass(in_dim, 64, num_classes, dropout=0.0).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()

    # Ensure all data on CPU before DataLoader (DataLoader moves to device per batch)
    clean_data = []
    for d in train_data:
        d = d.cpu()
        d.y = d.y.long()
        clean_data.append(d)

    loader = DataLoader(clean_data, batch_size=8, shuffle=True)

    model.train()
    for epoch in range(500):
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch.x, batch.edge_index, batch.batch)
            loss = loss_fn(pred, batch.y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

    # Diagnostic: check if surrogate collapsed to single class
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch.x, batch.edge_index, batch.batch)
            preds.extend(pred.argmax(1).cpu().tolist())
    unique, counts = np.unique(preds, return_counts=True)
    print(f"    Surrogate predictions: {dict(zip(unique, counts))}")

    return model

In [ ]:
# Balanced evaluation: equal samples per victim-predicted class

def get_balanced_test_idx(victim, dataset, test_idx, device='cuda', seed=42):
    by_class = {}
    victim.eval()
    with torch.no_grad():
        for idx in test_idx:
            data = dataset[idx].to(device)
            batch = torch.zeros(data.num_nodes, dtype=torch.long, device=device)
            pred = victim(data.x, data.edge_index, batch).argmax(1).item()
            by_class.setdefault(pred, []).append(idx)

    min_count = min(len(v) for v in by_class.values())
    rng = np.random.RandomState(seed)
    balanced_idx = []
    for cls in sorted(by_class.keys()):
        chosen = rng.choice(by_class[cls], min_count, replace=False)
        balanced_idx.extend(chosen)

    return balanced_idx

def evaluate(surrogate, victim, dataset, balanced_idx, device='cuda'):
    surrogate.eval()
    victim.eval()

    correct_fidelity = 0
    correct_accuracy = 0
    total = 0

    with torch.no_grad():
        for idx in balanced_idx:
            data = dataset[idx].to(device)
            batch = torch.zeros(data.num_nodes, dtype=torch.long, device=device)

            surr_pred = surrogate(data.x, data.edge_index, batch).argmax(1).item()
            vic_pred = victim(data.x, data.edge_index, batch).argmax(1).item()
            true_label = data.y.item()

            if surr_pred == vic_pred:
                correct_fidelity += 1
            if surr_pred == true_label:
                correct_accuracy += 1
            total += 1

    return round(correct_fidelity / total, 4), round(correct_accuracy / total, 4)

## Results Manager

In [ ]:
ABLATION_RESULTS_FILE = os.path.join(RESULTS_DIR, 'ablation_results.json')

def load_ablation_results():
    if os.path.exists(ABLATION_RESULTS_FILE):
        with open(ABLATION_RESULTS_FILE, 'r') as f:
            return json.load(f)
    return {}

def save_ablation_result(results, key, value):
    results[key] = value
    with open(ABLATION_RESULTS_FILE, 'w') as f:
        json.dump(results, f, indent=2)

## Run NoEXP and NoMC Ablations

In [ ]:
ABLATION_CONFIG = {
    'datasets': ['AIDS', 'NCI1', 'Tox21_AhR_training'],
    'architectures': ['GCN'],
    'explainers': ['GNN'],
    'methods': ['NoMC'],
    'budgets': [0.70],
}

def run_ablation_experiments(config):
    results = load_ablation_results()

    for name in config['datasets']:
        ds = datasets[name]
        splits = dataset_splits[name]
        shadow_idx = splits['shadow']
        test_idx = splits['test']
        in_dim = get_feature_dim(ds)
        num_classes = ds.num_classes

        for arch in config['architectures']:
            key_prefix = f"{name}_{arch}"
            if key_prefix not in victim_models:
                print(f"  SKIP: {key_prefix} not loaded")
                continue

            victim = victim_models[key_prefix]
            balanced_test = get_balanced_test_idx(victim, ds, test_idx, device)

            # Phase 1 (shared, computed once per dataset-arch pair)
            print(f"\n{'='*60}")
            print(f"Ablations: {arch} on {name}")
            print(f"{'='*60}")
            pairs_p1, non_bd_idx = find_boundary_pairs_exhaustive(
                ds, shadow_idx, victim, device)

            for budget in config['budgets']:
                n_samples = int(len(shadow_idx) * budget)
                budget_str = f"{int(budget*100)}pct"

                for method in config['methods']:
                    exp_configs = []
                    for exp_name in config['explainers']:
                        if exp_name == 'GNN':
                            exp = create_gnn_explainer(victim)
                        else:
                            continue
                        if exp is not None:
                            exp_configs.append((exp, exp_name))

                    for explainer, exp_name in exp_configs:
                        result_key = f"{name}_{arch}_{method}_{exp_name}_{budget_str}"

                        if result_key in results:
                            print(f"  SKIP (cached): {result_key}")
                            continue

                        print(f"\n--- {result_key} ---")

                        try:
                            train_data = method_boundary_no_mc(
                                ds, shadow_idx, victim, explainer,
                                n_samples, pairs_p1, non_bd_idx, device)
                        except Exception as e:
                            print(f"  ERROR: {e}")
                            continue

                        if len(train_data) == 0:
                            print(f"  No training data, skipping")
                            continue

                        print(f"  Training surrogate ({len(train_data)} samples)...")
                        surrogate = train_surrogate(
                            train_data, in_dim, num_classes, arch, device)

                        fid, acc = evaluate(
                            surrogate, victim, ds, balanced_test, device)

                        result = {
                            'fidelity': fid,
                            'accuracy': acc,
                            'n_train': len(train_data),
                            'timestamp': datetime.now().isoformat(),
                        }

                        print(f"  Fidelity: {fid:.4f}, Accuracy: {acc:.4f}, n_train: {len(train_data)}")
                        save_ablation_result(results, result_key, result)

    print(f"\nAll ablation results saved to {ABLATION_RESULTS_FILE}")
    return results

# Run it
ablation_results = run_ablation_experiments(ABLATION_CONFIG)